In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle


def plot_mini_cones_layout(
    ra=150.1191,
    dec=2.2058,
    radius=3600.0,  # arcsec, cone principal
):
    # Meme logique que ton code
    tile_radius = min(600.0, max(250.0, radius / 3.5))  # arcsec
    step_deg = 0.80 * tile_radius / 3600.0
    offs = (-1.0, 0.0, 1.0)

    fig, ax = plt.subplots(figsize=(7, 7))

    # Cone principal
    main_r_deg = radius / 3600.0
    ax.add_patch(Circle((ra, dec), main_r_deg, fill=False, lw=2, ec="black", label="Cone principal"))

    # 9 mini-cones
    first = True
    for dx in offs:
        for dy in offs:
            ra_i = ra + dx * step_deg
            dec_i = dec + dy * step_deg
            tile_r_deg = tile_radius / 3600.0

            ax.add_patch(
                Circle(
                    (ra_i, dec_i),
                    tile_r_deg,
                    fill=False,
                    lw=1.5,
                    ec="tab:blue",
                    alpha=0.8,
                    label="Mini-cones 3x3" if first else None,
                )
            )
            ax.plot(ra_i, dec_i, "o", ms=3, color="tab:red")
            first = False

    ax.plot(ra, dec, "x", ms=8, mew=2, color="black", label="Centre DDF")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title("Juxtaposition des mini-cones (fallback spatial 3x3)")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    print(f'radius principal = {radius:.1f}" ({main_r_deg:.4f} deg)')
    print(f'tile_radius      = {tile_radius:.1f}" ({tile_radius / 3600.0:.4f} deg)')
    print(f"step_deg         = {step_deg:.5f} deg")


# Exemple COSMOS
plot_mini_cones_layout(ra=150.1191, dec=2.2058, radius=3600.0)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle


def plot_adaptive_tiling(ra=150.1191, dec=2.2058, radius=3600.0):
    # Grand cone (arcsec)
    R = float(radius)

    # Rayon mini-cone (meme borne que ton code)
    rt = min(600.0, max(250.0, R / 3.5))

    # Pas max pour couvrir sans trou (avec marge de recouvrement)
    # condition de couverture locale: step <= sqrt(2) * rt
    step_arcsec = 0.95 * math.sqrt(2) * rt

    # Nombre de points par axe pour couvrir [-R, +R]
    n = int(math.ceil((2 * R) / step_arcsec)) + 1
    if n % 2 == 0:
        n += 1  # garder un centre exact

    offsets_arcsec = np.linspace(-R, R, n)
    step_deg = step_arcsec / 3600.0
    rt_deg = rt / 3600.0
    R_deg = R / 3600.0

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.add_patch(Circle((ra, dec), R_deg, fill=False, lw=2, ec="black", label="Cone principal"))

    first = True
    for dx_as in offsets_arcsec:
        for dy_as in offsets_arcsec:
            ra_i = ra + dx_as / 3600.0
            dec_i = dec + dy_as / 3600.0
            # option: ne garder que les centres utiles (dans le grand cone + marge)
            if math.hypot(dx_as, dy_as) > (R + rt):
                continue
            ax.add_patch(
                Circle(
                    (ra_i, dec_i),
                    rt_deg,
                    fill=False,
                    lw=1,
                    ec="tab:blue",
                    alpha=0.6,
                    label="Mini-cones" if first else None,
                )
            )
            first = False

    ax.plot(ra, dec, "x", color="black", ms=8, mew=2, label="Centre")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.set_title(f'Tiling adaptatif couvrant le cone initial (R={R:.0f}")')
    ax.grid(alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

    print(f'R={R:.0f}", rt={rt:.0f}", step~{step_arcsec:.0f}", grille {n}x{n}')


plot_adaptive_tiling()